In [ ]:
# Cell 1: imports and project-root setup
import os
import sys
from pathlib import Path

def find_project_root(start: Path) -> Path:
    """Find the project root by searching for a directory that contains `src`."""
    start = start.resolve()
    candidates = [start] + list(start.parents)
    for candidate in candidates:
        if (candidate / "src").exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import numpy as np
import torch
from torch.utils.data import DataLoader
import gym
import d4rl  # noqa: F401

from src.experiment_config import *
from src.config import CHECKPOINTS_DIR, RAW_METRICS_DIR, OBS_STATS_DIR
from src.dataset import NoisyOfflineRLDataset
from src.iql import IQLAgent
from src.train_eval import eval_policy_on_env


In [ ]:
# Cell 2: experiment configuration

METHOD = "raw_noisy"

def noise_tag(noise_dim, noise_scale, noise_type):
    """Build a compact tag used in checkpoint and metrics directories."""
    ns = str(noise_scale).replace(".", "p")
    return f"nd{noise_dim}_ns{ns}_{noise_type}"

NOISE_TAG = noise_tag(NOISE_DIM, NOISE_SCALE, NOISE_TYPE)
SEED_TAG = f"seed_{SEED}"

# Save checkpoints, metrics, and observation statistics into separate directories.
CKPT_DIR = CHECKPOINTS_DIR / METHOD / ENV_NAME / NOISE_TAG / SEED_TAG
METRICS_DIR = RAW_METRICS_DIR / METHOD / ENV_NAME / NOISE_TAG / SEED_TAG
OBS_STATS_PATH = OBS_STATS_DIR / METHOD / ENV_NAME / NOISE_TAG / SEED_TAG / "obs_stats.npz"

CKPT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
OBS_STATS_PATH.parent.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DEVICE:", DEVICE)
print("CKPT_DIR:", CKPT_DIR)
print("METRICS_DIR:", METRICS_DIR)
print("OBS_STATS_PATH:", OBS_STATS_PATH)


In [ ]:
# Cell 3: dataset and dataloader

dataset = NoisyOfflineRLDataset(
    env_name=ENV_NAME,
    noise_dim=NOISE_DIM,
    noise_scale=NOISE_SCALE,
    seed=SEED,
    use_timeouts=True,
    noise_type=NOISE_TYPE,
)

train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
                          num_workers=4, pin_memory=True, persistent_workers=True)

# Infer dimensions directly from the dataset tensors.
state_dim = dataset.noisy_obs.shape[1]
action_dim = dataset.actions.shape[1]
true_state_dim = dataset.obs_dim

print("state_dim (noisy):", state_dim)
print("true_state_dim:", true_state_dim)
print("action_dim:", action_dim)

# Persist normalization statistics for later online evaluation.
if not OBS_STATS_PATH.exists():
    np.savez(
        OBS_STATS_PATH,
        obs_mean=dataset.obs_mean,
        obs_std=dataset.obs_std,
        true_state_dim=true_state_dim,
    )
    print("Saved:", OBS_STATS_PATH)
else:
    print("Already exists:", OBS_STATS_PATH)


In [ ]:
# Cell 4: train IQL directly on noisy observations

iql = IQLAgent(
    latent_dim=state_dim,
    action_dim=action_dim,
    device=DEVICE,
    expectile=0.7,
    temperature=3.0,
    discount=0.99,
)

for epoch in range(1, EPOCHS + 1):
    v_losses, q_losses, a_losses = [], [], []

    # Dataset format:
    # (obs, act, next_obs, rew, done, pure_obs, pure_next_obs)
    for obs, act, next_obs, rew, done, pure_obs, pure_next_obs in train_loader:
        obs = obs.to(DEVICE)
        act = act.to(DEVICE)
        next_obs = next_obs.to(DEVICE)
        rew = rew.to(DEVICE)
        done = done.to(DEVICE)

        # Raw-noisy mode uses the corrupted observation directly as the latent state.
        z = obs
        next_z = next_obs

        v_loss, q_loss, a_loss = iql.train_step(z, act, next_z, rew, done)
        v_losses.append(v_loss)
        q_losses.append(q_loss)
        a_losses.append(a_loss)

    print(
        f"[IQL][{METHOD}] epoch {epoch}: "
        f"V={np.mean(v_losses):.4f}, "
        f"Q={np.mean(q_losses):.4f}, "
        f"A={np.mean(a_losses):.4f}"
    )

    if epoch % 10 == 0:
        ckpt_path = CKPT_DIR / f"iql_epoch_{epoch}.pth"
        torch.save(
            {
                "actor": iql.actor.state_dict(),
                "q_net": iql.q_net.state_dict(),
                "v_net": iql.v_net.state_dict(),
            },
            ckpt_path,
        )
        print("Saved:", ckpt_path)


In [ ]:
# Cell 5: evaluate the trained policy

metrics = eval_policy_on_env(
    iql=iql,
    env_name=ENV_NAME,
    encoder=None,
    method=METHOD,
    obs_mean=dataset.obs_mean,
    obs_std=dataset.obs_std,
    true_state_dim=true_state_dim,
    noise_dim=NOISE_DIM,
    noise_scale=NOISE_SCALE,
    noise_type=NOISE_TYPE,
    episodes=20,
    max_steps=1000,
    seed=SEED,
    device=DEVICE,
    use_fixed_noise=True,
)

metrics_path = METRICS_DIR / "metrics.json"
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "env": ENV_NAME,
            "method": METHOD,
            "seed": SEED,
            "noise_dim": NOISE_DIM,
            "noise_scale": NOISE_SCALE,
            "noise_type": NOISE_TYPE,
            **metrics,
        },
        f,
        indent=2,
    )

print("Saved metrics:", metrics_path)
print(metrics)
